<a href="https://colab.research.google.com/github/franciscomartinez45/Stock-Forecasting-LSTM/blob/main/notebooks/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [ ]:
!pip install -q transformers torch yfinance

In [ ]:
import os
from google.colab import userdata

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

HF_TOKEN = userdata.get('HF_TOKEN')
POLYGON_API_KEY = userdata.get('POLYGON_API_KEY')

In [ ]:
import time
import requests
import pandas as pd
import yfinance as yf
from transformers import pipeline

In [ ]:
def fetch_news(ticker, df_prices, api_key):
    start = df_prices.index[0].strftime("%Y-%m-%d")
    end   = df_prices.index[-1].strftime("%Y-%m-%d")
    url, articles, next_url, page = "https://api.polygon.io/v2/reference/news", [], None, 0
    params = {"ticker": ticker, "published_utc.gte": start, "published_utc.lte": end,
              "order": "asc", "limit": 1000, "apiKey": api_key}
    while True:
        resp = requests.get(next_url or url, params=params if not next_url else {"apiKey": api_key})
        batch = resp.json().get("results", [])
        articles.extend(batch)
        page += 1
        print(f"Page {page}: {len(batch)} articles (total: {len(articles)})")
        next_url = resp.json().get("next_url")
        if not next_url:
            break
        time.sleep(12)  # free-tier limit: 5 req/min
    return pd.DataFrame([{"date": a["published_utc"][:10], "title": a["title"],
                           "publisher": a["publisher"]["name"], "url": a["article_url"]}
                          for a in articles])


def score_daily_sentiment(pipe, df_prices, df_news, ticker):
    trading_dates = df_prices.index.strftime("%Y-%m-%d").tolist()
    rows = []
    for i, date in enumerate(trading_dates[:-1]):
        next_date  = trading_dates[i + 1]
        day_titles = df_news[df_news["date"] == date]["title"].tolist()
        if not day_titles:
            print(f"{date}: no articles, skipping")
            continue
        results = pipe(day_titles, batch_size=16, truncation=True)
        neg, neu, pos = [], [], []
        for scores in results:
            s = {r["label"].lower(): r["score"] for r in scores}
            neg.append(s.get("negative", 0))
            neu.append(s.get("neutral",  0))
            pos.append(s.get("positive", 0))
        close_today = df_prices.loc[date,      "Close"].item()
        close_next  = df_prices.loc[next_date, "Close"].item()
        change_pct  = (close_next - close_today) / close_today * 100
        rows.append({"date": date, "n_articles": len(day_titles),
                     "avg_negative": round(sum(neg)/len(neg), 4),
                     "avg_neutral":  round(sum(neu)/len(neu), 4),
                     "avg_positive": round(sum(pos)/len(pos), 4),
                     "close":        round(close_today, 2),
                     "next_close":   round(close_next,  2),
                     "next_day_pct": round(change_pct,  4),
                     "direction":    "UP" if close_next > close_today else "DOWN"})
        r = rows[-1]
        print(f"{date}: {r['n_articles']} articles | neg={r['avg_negative']:.3f} "
              f"neu={r['avg_neutral']:.3f} pos={r['avg_positive']:.3f} | "
              f"{r['direction']} ({change_pct:+.2f}%)")
    return pd.DataFrame(rows)

## Model Validation

In [ ]:
!wget -q "https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip" -O financial_phrasebank.zip
!unzip -q financial_phrasebank.zip

In [ ]:
label_map_str = {"negative": 0, "neutral": 1, "positive": 2}
label_map_int = {0: "negative", 1: "neutral", 2: "positive"}
sentences, true_labels = [], []

with open("FinancialPhraseBank-v1.0/Sentences_AllAgree.txt", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if "@" in line:
            sentence, label = line.rsplit("@", 1)
            sentences.append(sentence.strip())
            true_labels.append(label_map_str[label.strip()])

print(f"Loaded {len(sentences)} samples")

val_pipe = pipeline("text-classification", model="ProsusAI/finbert", device=0, top_k=None)
results  = val_pipe(sentences[:200], batch_size=16, truncation=True)

rows = []
for sentence, true_label, scores in zip(sentences[:200], true_labels[:200], results):
    score_dict = {s["label"].lower(): round(s["score"], 4) for s in scores}
    rows.append({"sentence": sentence, "true_label": label_map_int[true_label],
                 **score_dict, "predicted_label": max(score_dict, key=score_dict.get)})

df_val = pd.DataFrame(rows)
accuracy = (df_val["true_label"] == df_val["predicted_label"]).mean()
print(f"Accuracy on {len(df_val)} samples: {accuracy:.2%}")
print(df_val[["sentence", "true_label", "negative", "neutral", "positive", "predicted_label"]].head(10).to_string(index=False))

## Data Collection
### Apple (AAPL)

In [ ]:
df_aapl = yf.download("AAPL", start="2023-01-01", end="2024-01-01", auto_adjust=True)
print(f"Shape: {df_aapl.shape} | {df_aapl.index[0].date()} → {df_aapl.index[-1].date()}")
print(df_aapl.head())

In [ ]:
df_aapl_news = fetch_news("AAPL", df_aapl, POLYGON_API_KEY)
df_aapl_news.to_csv("data/raw/aapl_news_2023.csv", index=False)
print(f"\n{len(df_aapl_news)} articles saved to data/raw/aapl_news_2023.csv")
print(df_aapl_news.head(10).to_string(index=False))

### Microsoft (MSFT)

In [ ]:
df_msft = yf.download("MSFT", start="2023-01-01", end="2024-01-01", auto_adjust=True)
print(f"Shape: {df_msft.shape} | {df_msft.index[0].date()} → {df_msft.index[-1].date()}")
print(df_msft.head())

In [ ]:
df_msft_news = fetch_news("MSFT", df_msft, POLYGON_API_KEY)
df_msft_news.to_csv("data/raw/msft_news_2023.csv", index=False)
print(f"\n{len(df_msft_news)} articles saved to data/raw/msft_news_2023.csv")
print(df_msft_news.head(10).to_string(index=False))

## Sentiment Scoring

In [ ]:
pipe = pipeline("text-classification", model="ProsusAI/finbert", device=0, top_k=None)

### Apple (AAPL)

In [ ]:
df_aapl_daily = score_daily_sentiment(pipe, df_aapl, df_aapl_news, "AAPL")
df_aapl_daily.to_csv("data/processed/aapl_daily_sentiment_2023.csv", index=False)
print(f"\nDone. {len(df_aapl_daily)} days | saved to data/processed/aapl_daily_sentiment_2023.csv")
print(df_aapl_daily.head(10).to_string(index=False))

### Microsoft (MSFT)

In [ ]:
df_msft_daily = score_daily_sentiment(pipe, df_msft, df_msft_news, "MSFT")
df_msft_daily.to_csv("data/processed/msft_daily_sentiment_2023.csv", index=False)
print(f"\nDone. {len(df_msft_daily)} days | saved to data/processed/msft_daily_sentiment_2023.csv")
print(df_msft_daily.head(10).to_string(index=False))